# Baseline Atlas-DeePC: control-parameter ablation

Sweep **λg only** using the graph-waypoint algorithm in `baseline_graph_routed_atlas_deepc.ipynb`: **Clarabel**, 48 fixed physical-configuration charts, **rank-60 uncentered linear bases**, and iterative minimum-route-cost terminal-reference selection with **N=10** candidates. The comparison point is explicitly **λg = 0.05**.

The sweep uses **100 paired seeds, 211–310**, zero, 50 log-spaced positive values from 0.01 to 100, and the exact baseline 0.05 (**52 values / 5,200 full 320-step rollouts**). No results have been generated by this update. Historical results are not used.

## Run on your computer

1. Copy the project folder, including `src/`, this notebook, `data/select_iid_offline_hankel.npy`, and `experiments/routed_deepc_iid/clustering_jointconfig48_fresh211_310/behavioral_atlas.npz`. You do not need historical rollout folders or the atlas folder's old `summary.json`.
2. In a **Python 3.10** environment, from the project root run:
   ```text
   python -m pip install -r requirements-control-parameter.txt
   python -m ipykernel install --user --name atlas-deepc --display-name "Python (Atlas-DeePC)"
   python -m jupyterlab
   ```
   Select **Python (Atlas-DeePC)** in Jupyter or VS Code. Other operating systems may produce slightly different numerical results; recorded package versions and data hashes identify the run.
3. Run the setup/design cells. `PROJECT_ROOT=None` discovers the project from the current folder; set it to your local project path if needed. No computer-specific paths are embedded.
4. Set **`RUN_EXPERIMENTS=True`** when ready, then run the experiment cell. Default **`WORKERS=1`** gives serial timing; raise it for throughput if desired. The default sweep contains 5,200 rollouts; runtime depends on your computer and parameter values. All workers use one numerical thread. No rendering or GPU is required.
5. Interrupted runs resume from saved per-rollout checkpoints with identical settings. For a smaller pilot, change the seeds/grid and use a **different `OUTPUT_ROOT`**. To inspect saved results, keep `RUN_EXPERIMENTS=False` and run all cells.

The notebook starts with experiments disabled. It launches an ordinary Python script with the selected kernel's interpreter, so optional multiprocessing does not depend on notebook-defined workers. Stop the script before starting another run; the result-folder lock prevents concurrent writers.


In [ ]:
import json
import math
import os
import sys
import tempfile
from dataclasses import asdict
from pathlib import Path

PROJECT_ROOT = None  # Example: Path("D:/research/atlas_deepc_reacher")
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "atlas_deepc_mpl"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

if PROJECT_ROOT is None:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src/baseline_control_parameter_ablation.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Set PROJECT_ROOT to your copied atlas_deepc_reacher folder.")
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
sys.path.insert(0, str(ROOT / "src"))
import baseline_control_parameter_ablation as ablation

DATA_PATH = ROOT / "data/select_iid_offline_hankel.npy"
ATLAS_PATH = ROOT / "experiments/routed_deepc_iid/clustering_jointconfig48_fresh211_310/behavioral_atlas.npz"
OUTPUT_ROOT = ROOT / "experiments/routed_deepc_iid/ablation_control_parameter_baseline_clarabel_100_v1"
for path in (DATA_PATH, ATLAS_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Copy the baseline input file to: {path}")
print("Project:", ROOT)
print("Python:", sys.executable)
print("Results:", OUTPUT_ROOT)


## 1. Controlled sweep design

The baseline objective includes λg‖g‖². Only this weight changes. Every value uses the same physical targets, offline columns, chart memberships, rank-60 basis matrices, coefficient bounds, graph routes, Q/R weights, constraints, 320-step horizon and success test. The online trajectory and selected references can change in response to λg.

Zero is included in the same sweep and shown on a symmetric-log axis. The exact 0.05 baseline is added to the 50-point positive grid. The default seed panel is retained from the old notebook; it overlaps development studies and is not an untouched test set. The preflight below checks files, solver availability and hashes only; it does not simulate an environment.


In [ ]:
LOG_GRID_SIZE = 50
EVALUATION_SEEDS = list(range(211, 311))  # 100 seeds; stop is exclusive.
RUN_EXPERIMENTS = True  # Change to True on your computer to run/resume.
WORKERS = 1  # Optional: increase for throughput; parallel timings reflect contention.

BASE_CONFIG = ablation.default_config()  # Explicit lam_g=0.05, matching the notebook.
BASELINE_LAM_G = 0.05
POSITIVE_LAM_G_VALUES = np.unique(np.r_[np.logspace(-2, 2, LOG_GRID_SIZE), BASELINE_LAM_G])
LAM_G_VALUES = np.r_[0.0, POSITIVE_LAM_G_VALUES]
assert BASE_CONFIG.lam_g == BASELINE_LAM_G
SIGNATURE = ablation.make_signature(LAM_G_VALUES, EVALUATION_SEEDS, DATA_PATH, ATLAS_PATH, WORKERS)
SETTINGS = {"data_path": str(DATA_PATH), "atlas_path": str(ATLAS_PATH),
            "output_root": str(OUTPUT_ROOT), "lam_g_values": LAM_G_VALUES.tolist(),
            "seeds": EVALUATION_SEEDS, "workers": WORKERS}
display(pd.Series({
    "baseline lam_g": BASELINE_LAM_G, "solver": "CLARABEL (max_iter=10000)",
    "chart geometry": "uncentered linear subspace", "charts": 48, "rank": 60,
    "terminal reference": "iterative N=10, minimum route cost", "graph waypoints": True,
    "parameter values": len(LAM_G_VALUES), "seeds per value": len(EVALUATION_SEEDS),
    "seed range": f"{min(EVALUATION_SEEDS)}–{max(EVALUATION_SEEDS)}",
    "rollouts": len(LAM_G_VALUES)*len(EVALUATION_SEEDS), "steps each": 320,
    "workers": WORKERS, "numerical threads per worker": 1,
    "offline hashes": "verified", "experiments enabled": RUN_EXPERIMENTS,
}))
display(pd.DataFrame({"lam_g": LAM_G_VALUES}).style.format({"lam_g": "{:.8g}"}))


## 2. Run or resume

Each worker loads the immutable data and builds the baseline chart matrices and route tables once. A 10-step seed-210 warmup per worker is excluded. Lambda order rotates within each seed. A new versioned directory prevents accidentally reusing the old affine/fixed-reference results. Every completed rollout is saved atomically; a failed or interrupted rollout remains pending. Solver failures retain the baseline's zero-action behavior and count in the result.

Resume checks the exact seed/grid, worker count, controller settings, source hashes, package versions and offline hashes. Moving the project folder does not invalidate results. Changed settings require a new output folder; there is no overwrite switch. Compact JSON checkpoints retain configuration, target, errors, costs, solver counts and timing statistics; temporary full trajectories are removed after extracting those metrics.

The subprocess prints progress every 25 saved rollouts. Its settings JSON also supports command-line execution: `python src/baseline_control_parameter_ablation.py --settings <path-to-run_settings.json>`.


Every launch saves a timestamped `run_*.log`. If the subprocess fails, its last 40 output lines and log path are included in the exception.

In [ ]:
if RUN_EXPERIMENTS:
    import subprocess
    import signal
    from collections import deque
    from datetime import datetime, timezone
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    settings_path = OUTPUT_ROOT / "run_settings.json"
    ablation.write_json(settings_path, SETTINGS)
    command = [sys.executable, "-u", str(ROOT / "src/baseline_control_parameter_ablation.py"),
               "--settings", str(settings_path)]
    options = {"start_new_session": True} if os.name != "nt" else {}
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, **options)
    log_path = OUTPUT_ROOT / ("run_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
    recent_output = deque(maxlen=40)
    print("Full run log:", log_path)
    try:
        with log_path.open("w", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                recent_output.append(line)
                print(line, end="", flush=True)
        return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(
                f"Sweep exited with code {return_code}. Valid checkpoints are retained. "
                f"Full log: {log_path}\n\nLast subprocess output:\n" + "".join(recent_output)
            )
    except KeyboardInterrupt:
        if os.name != "nt":
            os.killpg(process.pid, signal.SIGTERM)
        else:
            subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], check=False)
        process.wait()
        print("Stopped. Rerun with unchanged settings to resume saved rollouts.")
        raise
else:
    print(f"Experiments disabled: {len(LAM_G_VALUES)*len(EVALUATION_SEEDS):,} rollouts configured.")


## 3. Completion and controlled-change audit

Read checkpoints directly, including partially completed panels. Configurations and physical targets must agree across paired seeds. Only complete panels enter performance curves and paired comparisons. A partial sweep never substitutes unmatched seed panels into the plots.


In [ ]:
manifest_path = OUTPUT_ROOT / "manifest.json"
payloads = {}
if manifest_path.exists():
    manifest, payloads = ablation.load_lam_g_sweep(OUTPUT_ROOT)
    if manifest["signature"] != SIGNATURE:
        raise ValueError("Saved results do not match this notebook's settings, sources or packages.")
completion_rows, valid_payloads = [], {}
for lam_g in LAM_G_VALUES:
    payload = payloads.get(float(lam_g))
    saved = len(payload["results"]) if payload else 0
    complete = bool(payload and payload["complete"])
    completion_rows.append({"lam_g": lam_g, "saved": saved,
                            "missing": len(EVALUATION_SEEDS)-saved, "complete": complete})
    if complete:
        valid_payloads[float(lam_g)] = payload
completion = pd.DataFrame(completion_rows)
display(completion.style.format({"lam_g": "{:.8g}"}))
complete_values = completion.loc[completion["complete"], "lam_g"].to_numpy()
print(f"Complete parameter values: {len(complete_values)}/{len(LAM_G_VALUES)}")
if not manifest_path.exists():
    print("No results yet. Enable RUN_EXPERIMENTS when ready to run on your computer.")


## 4. Performance over λg

The common physical-target cost is Σₜ[Qxy‖pₜ−p_target‖² + R‖uₜ‖²]. It measures fingertip tracking and effort against the same environment target for every lambda. It excludes joint-reference penalties, slack, coefficient regularization and input-increment penalties; it is an evaluation metric, not the complete QP objective. The checkpoint also retains the historical full-Q costs against the active reference and the last selected terminal joint configuration, which can differ across lambda values and are reference-dependent diagnostics.

Strict success requires **both** final-30 and final-100 mean errors below 10 mm and maximum errors below 20 mm. Error/cost distributions include failed trials. Success bands are 95% Wilson intervals. The timing panel is the median across rollout-level medians, after discarding each rollout's first five steps; it is not a pooled per-step median. Setup and trace writing are excluded from online timing. Parallel workers and shared-computer load affect timings; do not compare them directly with prior serial measurements.


In [ ]:
def strict_success(row):
    return bool(row["success_30"] and row["success_100"])

def wilson_interval(successes, trials, z=1.959963984540054):
    rate = successes / trials
    denominator = 1 + z * z / trials
    center = (rate + z * z / (2 * trials)) / denominator
    half_width = z * np.sqrt(rate * (1 - rate) / trials + z * z / (4 * trials**2)) / denominator
    return center - half_width, center + half_width

records = []
for lam_g, payload in valid_payloads.items():
    for row in payload["results"]:
        usable_solves = int(row["statuses"].get("optimal", 0) + row["statuses"].get("optimal_inaccurate", 0))
        records.append({
            "lam_g": lam_g,
            "seed": int(row["seed"]),
            "strict success": strict_success(row),
            "task cost": row["closed_loop_task_cost"],
            "active-reference cost": row["closed_loop_reference_cost"],
            "final-100 mean (mm)": 1000 * row["mean_last_100"],
            "prediction error (mm)": 1000 * row["mean_prediction_error_xy"],
            "usable solve fraction": usable_solves / BASE_CONFIG.steps,
            "median control time (ms)": row["median_control_compute_time_ms"],
        })
results = pd.DataFrame(records)

summary_rows = []
if not results.empty:
    for lam_g, subset in results.groupby("lam_g", sort=True):
        successes = int(subset["strict success"].sum())
        ci_low, ci_high = wilson_interval(successes, len(subset))
        summary_rows.append({
            "lam_g": lam_g, "seeds": len(subset), "successes": successes,
            "success rate": successes / len(subset), "CI low": ci_low, "CI high": ci_high,
            "task cost q25": subset["task cost"].quantile(0.25),
            "task cost median": subset["task cost"].median(),
            "task cost q75": subset["task cost"].quantile(0.75),
            "active-reference cost median": subset["active-reference cost"].median(),
            "final-100 q25 (mm)": subset["final-100 mean (mm)"].quantile(0.25),
            "final-100 median (mm)": subset["final-100 mean (mm)"].median(),
            "final-100 q75 (mm)": subset["final-100 mean (mm)"].quantile(0.75),
            "prediction error median (mm)": subset["prediction error (mm)"].median(),
            "usable solve rate": subset["usable solve fraction"].mean(),
            "control time median (ms)": subset["median control time (ms)"].median(),
        })
aggregate = pd.DataFrame(summary_rows).sort_values("lam_g") if summary_rows else pd.DataFrame()
if not aggregate.empty:
    display(aggregate.style.format({
        "lam_g": "{:.8g}", "success rate": "{:.1%}", "CI low": "{:.1%}", "CI high": "{:.1%}",
        "usable solve rate": "{:.2%}", "task cost median": "{:.5g}",
        "final-100 median (mm)": "{:.3f}", "prediction error median (mm)": "{:.3f}",
        "control time median (ms)": "{:.2f}",
    }))
else:
    print("No valid results are available yet.")

In [ ]:
if not aggregate.empty:
    plot_values = set(float(value) for value in complete_values)
    plot_data = aggregate[aggregate["lam_g"].isin(plot_values)].copy()
    x = plot_data["lam_g"].to_numpy()
    fig, axes = plt.subplots(2, 3, figsize=(18, 9.5), constrained_layout=True)

    axes[0, 0].plot(x, plot_data["task cost median"], marker="o", ms=3, color="#1d4ed8", label="physical-target + input cost")
    axes[0, 0].fill_between(x, plot_data["task cost q25"], plot_data["task cost q75"], color="#93c5fd", alpha=0.35, label="IQR")
    axes[0, 0].set_yscale("log")
    axes[0, 0].set_ylabel("physical-target + input cost")
    axes[0, 0].set_title("Common task cost (median and IQR)")
    axes[0, 0].legend(frameon=False, fontsize=9)

    axes[0, 1].plot(x, 100 * plot_data["success rate"], marker="o", ms=3, color="#15803d")
    axes[0, 1].fill_between(x, 100 * plot_data["CI low"], 100 * plot_data["CI high"], color="#86efac", alpha=0.35)
    axes[0, 1].set_ylim(0, 102)
    axes[0, 1].set_ylabel("strict success rate (%)")
    axes[0, 1].set_title("Strict success (95% Wilson interval)")

    axes[0, 2].plot(x, plot_data["final-100 median (mm)"], marker="o", ms=3, color="#c2410c")
    axes[0, 2].fill_between(x, plot_data["final-100 q25 (mm)"], plot_data["final-100 q75 (mm)"], color="#fdba74", alpha=0.35)
    axes[0, 2].axhline(10, color="#111827", linestyle="--", linewidth=1, label="10 mm mean threshold")
    axes[0, 2].set_yscale("log")
    axes[0, 2].set_ylabel("final-100 mean distance (mm)")
    axes[0, 2].set_title("Terminal accuracy (median and IQR)")
    axes[0, 2].legend(frameon=False, fontsize=9)

    axes[1, 0].plot(x, plot_data["prediction error median (mm)"], marker="o", ms=3, color="#0f766e")
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_ylabel("mean one-step XY error (mm)")
    axes[1, 0].set_title("Prediction accuracy")

    axes[1, 1].plot(x, 100 * plot_data["usable solve rate"], marker="o", ms=3, color="#be123c")
    axes[1, 1].set_ylim(0, 101)
    axes[1, 1].set_ylabel("optimal / optimal-inaccurate solves (%)")
    axes[1, 1].set_title("Solver availability")

    axes[1, 2].plot(x, plot_data["control time median (ms)"], marker="o", ms=3, color="#6d28d9")
    axes[1, 2].set_ylabel("median of rollout medians (ms/step)")
    axes[1, 2].set_title("Online control time (first 5 steps excluded)")

    symlog_threshold = 0.5 * POSITIVE_LAM_G_VALUES.min()
    lam_g_ticks = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]
    lam_g_ticklabels = ["0", r"$10^{-2}$", r"$10^{-1}$", r"$10^0$", r"$10^1$", r"$10^2$"]
    for axis in axes.flat:
        axis.set_xscale("symlog", linthresh=symlog_threshold, linscale=1.0)
        axis.set_xticks(lam_g_ticks)
        axis.set_xticklabels(lam_g_ticklabels)
        axis.set_xlabel(r"$\lambda_g$")
        axis.axvline(BASELINE_LAM_G, color="#111827", linestyle=":", linewidth=1.2)
        axis.grid(alpha=0.2, which="both")
    fig.suptitle(f"Baseline graph-routed Atlas-DeePC: lam_g ablation including zero ({len(plot_data)} values; {len(EVALUATION_SEEDS)} paired seeds each)")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    figure_path = OUTPUT_ROOT / "lam_g_ablation_performance.png"
    table_path = OUTPUT_ROOT / "lam_g_ablation_summary.csv"
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    aggregate.to_csv(table_path, index=False)
    plt.show()
    print("Saved:", figure_path)
    print("Saved:", table_path)
else:
    print("Plots will be generated after at least one complete lambda panel exists.")

## 5. Paired comparison with λg = 0.05

Compare complete panels on the identical seed set. The cost ratio uses the common physical-target + input metric defined above. The exact two-sided McNemar test uses discordant strict-success outcomes. These p-values are descriptive and unadjusted across the many tested lambda values; they are not a multiple-comparison-controlled selection rule.


In [ ]:
def exact_mcnemar_p(first_only, second_only):
    discordant = first_only + second_only
    if discordant == 0:
        return 1.0
    smaller = min(first_only, second_only)
    tail = sum(math.comb(discordant, k) for k in range(smaller + 1))
    return min(1.0, 2.0 * tail / (2 ** discordant))

paired_rows = []
if not results.empty:
    baseline_rows = results[np.isclose(results["lam_g"], BASELINE_LAM_G, rtol=0, atol=0)].set_index("seed")
    if not baseline_rows.empty:
        for lam_g, candidate_rows in results.groupby("lam_g", sort=True):
            candidate_rows = candidate_rows.set_index("seed")
            common = baseline_rows.index.intersection(candidate_rows.index)
            base = baseline_rows.loc[common]
            candidate = candidate_rows.loc[common]
            baseline_only = int((base["strict success"] & ~candidate["strict success"]).sum())
            candidate_only = int((~base["strict success"] & candidate["strict success"]).sum())
            paired_rows.append({
                "lam_g": lam_g, "paired seeds": len(common),
                "success-rate change": candidate["strict success"].mean() - base["strict success"].mean(),
                "candidate-only successes": candidate_only, "baseline-only successes": baseline_only,
                "exact McNemar p": exact_mcnemar_p(baseline_only, candidate_only),
                "median task cost ratio": np.median(candidate["task cost"] / base["task cost"]),
                "median final-100 change (mm)": np.median(candidate["final-100 mean (mm)"] - base["final-100 mean (mm)"]),
            })
paired = pd.DataFrame(paired_rows)
if paired_rows:
    display(paired.style.format({
        "lam_g": "{:.8g}", "success-rate change": "{:+.1%}", "exact McNemar p": "{:.4g}",
        "median task cost ratio": "{:.3f}x", "median final-100 change (mm)": "{:+.3f}",
    }))
    paired_path = OUTPUT_ROOT / "lam_g_ablation_paired_vs_baseline.csv"
    paired.to_csv(paired_path, index=False)
    print("Saved:", paired_path)
else:
    print("Paired comparisons require results at the exact baseline value lam_g=0.05.")

## Interpretation and saved artifacts

- Only λg changes. The algorithm remains rank-60 linear Atlas-DeePC with graph waypoints, iterative N=10 references, and Clarabel for every QP.
- Include failures in error and cost distributions. Solver exceptions count as unsolved steps under the baseline behavior; unexpected experiment exceptions stop execution and leave the rollout pending.
- Complete paired panels support comparison; partial panels are shown only in the completion table. The development seed panel overlaps prior studies.
- `manifest.json` records the protocol, data/source hashes and package versions. Per-lambda `seed*.json` files are atomic resumable checkpoints. `session_*.json` records each launch; plots and aggregate/paired tables are saved alongside them.
- Setup is performed only at experiment launch. No data collection, chart clustering, SVD fitting during online control, or reuse of historical timings occurs.
- Run the final analysis cells again after resuming. A fully completed sweep has 52 complete values and 5,200 validated results with the default settings.
